In [ ]:
import numpy as np
from matplotlib import pyplot as plt

%matplotlib inline
import taichi as ti

ti.init(
    arch=ti.cpu,
    default_fp=ti.f64,
    cpu_max_num_threads=1,
    offline_cache=False,
    debug=True,
)

from pespace.detector.antenna import InterferometerAntenna, FDResponseModelMarset2018
from pespace.detector.tdi import TDIChannelData, FDMichelsonConstantEqualArm
from pespace.detector.orbit import KaplerianHeliocentric
from tiwave.waveforms import IMRPhenomXAS

import lal
import bilby

## Frequency-domain response

In [ ]:
f_ref = 1e-4
f_min = 1e-4
f_max = 0.1
t_start = 0.0

channels = ("A", "E", "T")
delta_time = 5
num_tsamples = 2 ** np.ceil(np.log2(4 * lal.DAYJUL_SI / delta_time))
duration = num_tsamples * delta_time
before_tc = 0.8 * duration
after_tc = 0.2 * duration
tc = t_start + before_tc
# tc = 0.0
print("sample num: ", num_tsamples)
print("duration: ", duration)
print("tc: ", tc)

params = dict(
    total_mass=3e6,
    mass_ratio=0.6,
    chi1_z=0.75,
    chi2_z=0.62,
    luminosity_distance=56000.0,
    inclination=0.4,
    reference_phase=1.3,
    ecliptic_longitude=1.375,
    ecliptic_latitude=-1.2108,
    polarization=2.659,
    coalescence_time=tc,
)
params = bilby.gw.conversion.generate_mass_parameters(params)
print(params)

In [ ]:
tdi_data = TDIChannelData()
tdi_data.set_fd_data_from_zero(
    channels,
    duration,
    delta_time,
    start_time=t_start,
    minimum_frequency=f_min,
    maximum_frequency=f_max,
)
orbit_model = KaplerianHeliocentric(2.5e9, 0.0, 0.0)
response_model = FDResponseModelMarset2018()
tdi_combination = FDMichelsonConstantEqualArm(generation="2.0", orthogonal=True)

lisa = InterferometerAntenna(
    name="lisa",
    tdi_data=tdi_data,
    orbit_model=orbit_model,
    response_model=response_model,
    tdi_combination=tdi_combination,
)

waveform_tiw = IMRPhenomXAS(tdi_data.frequency_samples, f_ref)


In [ ]:
waveform_tiw.update_waveform(params)

lisa.update_detector_response(
    waveform_tiw.waveform_container,
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    params["coalescence_time"],
)

In [ ]:
# abs
plt.figure()
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["A"]),
    label="A",
)
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["E"]),
    label="E",
)
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["T"]),
    label="T",
)
plt.ylim(1e-28, 1e-16)
plt.xlim(f_min, f_max)
plt.legend()

# real part
plt.figure()
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["A"].real,
    label="A real",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["E"].real,
    label="E real",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["T"].real,
    label="T real",
)
plt.xlim(f_min, f_max)
plt.legend()

# imag part
plt.figure()
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["A"].imag,
    label="A imag",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["E"].imag,
    label="E imag",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["T"].imag,
    label="T imag",
)
plt.xlim(f_min, f_max)
plt.legend()

## Time-domain response